# Phase 14 Colab 5% HQ Runs, No Drive

Use this notebook when Google Drive is full, unavailable, or credential propagation fails. Everything runs in `/content`; download the split result files before stopping the runtime.

In [ ]:
from google.colab import files
import pathlib, shutil, yaml, os, json, zipfile, hashlib

ROOT = pathlib.Path('/content/ns_mc_gan_gi_phase14')
DATA_ROOT = pathlib.Path('/content/ns_mc_gan_gi_data')
OUT_ROOT = pathlib.Path('/content/outputs_phase14_colab')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)

print('Upload ns_mc_gan_gi_project_phase14_colab.zip now')
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
if not zip_names:
    raise RuntimeError('No zip uploaded')
ZIP = pathlib.Path('/content') / zip_names[0]
print('Using zip:', ZIP)

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__)
print('cuda available', torch.cuda.is_available())
print('gpu', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

In [ ]:
if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)
!unzip -q -o "$ZIP" -d /content/ns_mc_gan_gi_phase14
%cd /content/ns_mc_gan_gi_phase14
!pip -q install 'numpy<2' tqdm matplotlib scikit-image PyYAML tensorboard
!python -m compileall src

## Patch Configs To `/content`

This rewrites only the runtime copies in Colab. Your local repo configs stay unchanged.

In [ ]:
CONFIGS = {
    'rademacher5': pathlib.Path('configs/phase14_colab/rademacher5_hq_noise001_colab.yaml'),
    'scrambled5': pathlib.Path('configs/phase14_colab/scrambled_hadamard5_hq_noise001_colab.yaml'),
}
for key, cfg_path in CONFIGS.items():
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg['dataset_root'] = str(DATA_ROOT)
    cfg['output_dir'] = str(OUT_ROOT / cfg_path.stem)
    cfg['num_workers'] = 2
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
    pathlib.Path(cfg['output_dir']).mkdir(parents=True, exist_ok=True)
    print(cfg_path, '->', cfg['output_dir'])

## Run Rademacher 5%

In [ ]:
%cd /content/ns_mc_gan_gi_phase14
!python -m src.train --config configs/phase14_colab/rademacher5_hq_noise001_colab.yaml --device cuda
!python -m src.eval_auto --output_dir /content/outputs_phase14_colab/rademacher5_hq_noise001_colab --config configs/phase14_colab/rademacher5_hq_noise001_colab.yaml --device cuda
!python -m src.analyze_convergence --output_dir /content/outputs_phase14_colab/rademacher5_hq_noise001_colab

## Run Scrambled Hadamard 5%

In [ ]:
%cd /content/ns_mc_gan_gi_phase14
!python -m src.train --config configs/phase14_colab/scrambled_hadamard5_hq_noise001_colab.yaml --device cuda
!python -m src.eval_auto --output_dir /content/outputs_phase14_colab/scrambled_hadamard5_hq_noise001_colab --config configs/phase14_colab/scrambled_hadamard5_hq_noise001_colab.yaml --device cuda
!python -m src.analyze_convergence --output_dir /content/outputs_phase14_colab/scrambled_hadamard5_hq_noise001_colab

## Check Progress Or Finished Files

In [ ]:
!find /content/outputs_phase14_colab -maxdepth 2 \( -name 'eval_metrics.json' -o -name 'best_hq.pt' -o -name 'per_epoch_metrics.csv' \) -print | sort
!du -h -d 1 /content/outputs_phase14_colab || true

## Summarize, Package, Split

In [ ]:
%cd /content/ns_mc_gan_gi_phase14
!python -m src.phase14_colab_summary --base_dir /content/outputs_phase14_colab --output_dir /content/outputs_phase14_colab/_summary
!python -m src.package_colab_outputs --input_dir /content/outputs_phase14_colab --output_zip /content/phase14_colab_outputs.zip --manifest /content/phase14_colab_outputs_manifest.json

def split_file(path, chunk_mb=50):
    path = pathlib.Path(path)
    out_dir = pathlib.Path('/content/phase14_download_parts')
    if out_dir.exists():
        shutil.rmtree(out_dir)
    out_dir.mkdir(parents=True)
    chunk = chunk_mb * 1024 * 1024
    parts = []
    with path.open('rb') as f:
        idx = 0
        while True:
            data = f.read(chunk)
            if not data:
                break
            part = out_dir / f'{path.name}.part_{idx:03d}'
            part.write_bytes(data)
            parts.append(part.name)
            idx += 1
    h = hashlib.sha256(path.read_bytes()).hexdigest()
    manifest = {'source': str(path), 'sha256': h, 'chunk_mb': chunk_mb, 'parts': parts}
    (out_dir / 'phase14_split_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return out_dir

parts_dir = split_file('/content/phase14_colab_outputs.zip', 50)
!ls -lh /content/phase14_colab_outputs.zip /content/phase14_colab_outputs_manifest.json /content/phase14_download_parts | head -80

## Download Parts

Download the manifest first, then each part. Put all parts on the PC in `E:/ns_mc_gan_gi/colab_downloads`, then merge/import locally.

In [ ]:
from pathlib import Path
files.download('/content/phase14_colab_outputs_manifest.json')
files.download('/content/phase14_download_parts/phase14_split_manifest.json')
for part in sorted(Path('/content/phase14_download_parts').glob('phase14_colab_outputs.zip.part_*')):
    print('download', part.name, part.stat().st_size / 1024**2, 'MB')
    files.download(str(part))